# MODEL TRAINING: SageMaker's Image-Classifier (Transfer Learning) 

In [1]:
import boto3
import sagemaker
from sagemaker import get_execution_role

role = get_execution_role()
print(role)

region = boto3.Session().region_name

s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")

sess = sagemaker.Session()

# project bucket
bucket_name = "aai-540-data"

# image source and lst files
images_prefix = "cct_resized"
s3_images_location = f"s3://{bucket_name}/{images_prefix}/"
s3_train_lst = "s3://aai-540-data/dev_split/train.lst"
s3_validation_lst = "s3://aai-540-data/dev_split/validation/validation.lst"

# specifiy output location of training data and model
output_prefix = "sg-ic-transfer-learning"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
arn:aws:iam::324183265896:role/service-role/AmazonSageMaker-ExecutionRole-20250604T045982


In [2]:
# retrieve base SageMakers image-classification model 
from sagemaker import image_uris

training_image = image_uris.retrieve(
    framework = "image-classification", region = sess.boto_region_name, version="latest"
)
print(training_image)

811284229777.dkr.ecr.us-east-1.amazonaws.com/image-classification:1


In [3]:
# Configure input channels
input_data = {
    "train": sagemaker.inputs.TrainingInput(
        s3_data=s3_images_location,  
        content_type="application/x-image",
    ),
    "validation": sagemaker.inputs.TrainingInput(
        s3_data=s3_images_location,  # Same directory as training
        content_type="application/x-image",
    ),
    "train_lst": sagemaker.inputs.TrainingInput(
        #s3_data=s3_images_location + 'train_lst/' + 'train.lst',
        s3_data = s3_train_lst,
        content_type="application/x-image",
    ),
    "validation_lst": sagemaker.inputs.TrainingInput(
        #s3_data=s3_images_location + 'val_lst/' + 'val.lst',
        s3_data = s3_validation_lst,
        content_type="application/x-image",
    ),
}

In [4]:
# Configure base image classifier
s3_output_location = f"s3://{bucket_name}/{output_prefix}/output"
ic_estimator = sagemaker.estimator.Estimator(
    image_uri = training_image,
    role = role,
    instance_count=1,
    instance_type="ml.g4dn.xlarge",
    volume_size=50,
    max_run=360000,
    input_mode="File",
    output_path=s3_output_location,
    sagemaker_session=sess,
)

In [5]:
# MAKE SURE THAT NUM_CLASSES are UPDATED ACCORDING TO DECIDED TIME SPLIT
# Configure hyper parameters

ic_estimator.set_hyperparameters(
    num_layers=18, 
    use_pretrained_model=1,
    image_shape="3,224,224",
    num_classes=15, # hardcoded from notebook 02
    num_training_samples=27906, # hardcoded from notebook 02
    mini_batch_size=128,
    epochs=10,
    learning_rate=0.01,
    precision_dtype="float32",
    early_stopping=True
)


In [6]:
# fit estimator
ic_estimator.fit(inputs=input_data, logs=True)

INFO:sagemaker:Creating training-job with name: image-classification-2025-06-23-19-03-50-359


2025-06-23 19:03:52 Starting - Starting the training job...
2025-06-23 19:04:08 Starting - Preparing the instances for training...
2025-06-23 19:04:35 Downloading - Downloading input data............
2025-06-23 19:06:31 Downloading - Downloading the training image........Docker entrypoint called with argument(s): train
Running default environment configuration script
Nvidia gpu devices, drivers and cuda toolkit versions (only available on hosts with GPU):
Mon Jun 23 19:08:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |     

In [7]:
# Register fine-tuned model on sagemaker
from sagemaker.model import Model

model = Model(
    image_uri=ic_estimator.image_uri,
    model_data=ic_estimator.model_data,
    role=role,
    sagemaker_session=sess
)

registered_model = model.register(
    model_package_group_name='wildscan-image-classifiers',  
    content_types=['image/jpeg'],
    response_types=['application/json'],
    approval_status='PendingManualApproval',
    description='Image classifier trained 15 classes as in notebook2'
)

print(f"Model registered with ARN: {registered_model.model_package_arn}")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 │   sagemaker_session=sess                                                                  │
│    9 )                                                                                           │
│   10                                                                                             │
│ ❱ 11 registered_model = model.register(                                                          │
│   12 │   model_package_group_name='wildscan-image-classifiers',                                  │
│   13 │   content_types=['image/jpeg'],                                                           │
│   14 │   response_types=['application/json'],                                                    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: Model.register() got an unexpected keyword argument 'model_approval_status'

In [8]:
registered_model = model.register(
    model_package_group_name='wildscan-image-classifiers',  
    content_types=['image/jpeg'],
    response_types=['application/json'],
    approval_status='PendingManualApproval',
    description='Image classifier trained 15 classes as in notebook2'
)

print(f"Model registered with ARN: {registered_model.model_package_arn}")

Model registered with ARN: arn:aws:sagemaker:us-east-1:324183265896:model-package/wildscan-image-classifiers/1
